In [ ]:
# Productor de eventos, el waf tiene que detectar la situacion de que el usuario esta intentando hacer varios login y lo banea

import random
from datetime import datetime, timedelta
import pandas as pd
from collections import defaultdict

class EventProducer:
    def __init__(self):
        self.actions = ['view', 'click', 'login', 'purchase']
        self.devices = ['mobile', 'desktop', 'tablet']

    def start_stream(self, total_events=50):
        # Genera eventos usando yield
        current_time = datetime.now()
        for i in range(total_events):
            # Simula que los eventos ocurren con pocos segundos de diferencia
            current_time += timedelta(seconds=random.uniform(0.5, 2.0))

            # Fuerza que a veces el mismo usuario intente loguearse repetidamente
            if random.random() > 0.7:
                user_id = 42
                action = 'login'
            else:
                user_id = random.randint(1, 10)
                action = random.choice(self.actions)

            event = {
                'timestamp': current_time,
                'user_id': user_id,
                'action': action,
                'device': random.choice(self.devices),
                'ip': f"192.168.1.{user_id}"
            }
            yield event

class WAF:
    def __init__(self, max_logins=3, window_seconds=5):
        self.max_logins = max_logins
        self.window_seconds = window_seconds
        # Historial de logins
        self.login_history = defaultdict(list) # Para poder acceder a una clave que no existe aún sin comprobaciones
        self.banned_users = set()

    def process_event(self, event):
        # Procesa un evento y aplica reglas del WAF
        user_id = event['user_id']
        current_time = event['timestamp']

        # Si el usuario ya está baneado, bloquea cualquier acción
        if user_id in self.banned_users:
            return f"[BLOQUEADO] Intento de acción '{event['action']}' por usuario {user_id} (BANEADO)"

        if event['action'] == 'login':
            # Registra el intento actual
            self.login_history[user_id].append(current_time)

            # Filtra e identifica solo los intentos dentro de la ventana de tiempo
            cutoff_time = current_time - timedelta(seconds=self.window_seconds) # A partir de qué tiempo
            active_attempts = [t for t in self.login_history[user_id] if t > cutoff_time]
            self.login_history[user_id] = active_attempts
            # Verifica si se pasa del límite permitido
            if len(active_attempts) > self.max_logins:
                self.banned_users.add(user_id)
                return f"[BANEADO] El usuario {user_id} ha sido baneado. Intentos: {len(active_attempts)} en {self.window_seconds}s."

            return f"[LOGIN] Intento de login del usuario {user_id} (Intento {len(active_attempts)}/{self.max_logins})"

        return f"[OK] Usuario {user_id} realizó '{event['action']}'"

# SIMULACIÓN
print("Iniciando simulación...\n")

producer = EventProducer()

waf = WAF(max_logins=3, window_seconds=5)

for event in producer.start_stream(total_events=30):
    result = waf.process_event(event)
    timestamp_str = event['timestamp'].strftime('%H:%M:%S.%f')[:-3]
    print(f"[{timestamp_str}] {result}")

print(f"\nUsuarios baneados: {list(waf.banned_users)}")

Iniciando simulación...

[19:06:22.242] [LOGIN] Intento de login del usuario 42 (Intento 1/3)
[19:06:24.207] [OK] Usuario 10 realizó 'purchase'
[19:06:25.822] [OK] Usuario 9 realizó 'view'
[19:06:27.740] [OK] Usuario 10 realizó 'purchase'
[19:06:29.174] [LOGIN] Intento de login del usuario 42 (Intento 1/3)
[19:06:30.809] [OK] Usuario 10 realizó 'view'
[19:06:31.410] [OK] Usuario 8 realizó 'click'
[19:06:31.999] [LOGIN] Intento de login del usuario 2 (Intento 1/3)
[19:06:33.549] [LOGIN] Intento de login del usuario 42 (Intento 2/3)
[19:06:34.905] [LOGIN] Intento de login del usuario 5 (Intento 1/3)
[19:06:36.003] [LOGIN] Intento de login del usuario 7 (Intento 1/3)
[19:06:36.655] [OK] Usuario 6 realizó 'view'
[19:06:37.796] [LOGIN] Intento de login del usuario 42 (Intento 2/3)
[19:06:38.595] [LOGIN] Intento de login del usuario 42 (Intento 2/3)
[19:06:39.814] [OK] Usuario 10 realizó 'view'
[19:06:40.541] [OK] Usuario 1 realizó 'view'
[19:06:41.998] [LOGIN] Intento de login del usuario 9